In [28]:
import sys
import os

PROJECT_ROOT = os.path.dirname(os.getcwd())
sys.path.append(os.path.join(PROJECT_ROOT, "src"))

In [29]:
from conllu import parse_incr
from steps.projectivize import is_non_proj
import pandas as pd

In [30]:
def get_non_proj_sentences(gold_path):
    with open(gold_path, "r", encoding="utf-8") as f:
        sent_ids = []
        for tokenlist in parse_incr(f):
            arcs = []
            for token in tokenlist:
                arcs.append((token["head"], token["id"]))
            if is_non_proj(arcs):
                sent_ids.append(tokenlist.metadata["sent_id"])
    return sent_ids

In [32]:
gold_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
non_proj_sents = get_non_proj_sentences(gold_file)

In [49]:
def find_errors(gold_path, pred_path):

    errors = {
        "sent_id":[],
        "token_id": [], 
        "gold_head": [],
        "gold_head_pos":[], 
        "pred_head": [],
        "pred_head_pos":[], 
        "gold_deprel": [],
        "pred_deprel": []
        }    
 
    with open(pred_path, "r", encoding="utf-8") as fpred, \
         open(gold_path, "r", encoding="utf-8") as fgold:
        for sent_pred, sent_gold in zip(parse_incr(fpred), parse_incr(fgold)):
            sent_id = sent_gold.metadata["sent_id"]
            for tok_pred, tok_gold in zip(sent_pred, sent_gold):
                wrong_head = tok_pred["head"] != tok_gold["head"]
                wrong_deprel = tok_pred["deprel"] != tok_gold["deprel"]
                if wrong_head or wrong_deprel:
                    errors["sent_id"].append(sent_id)
                    errors["token_id"].append(tok_pred["id"])
                    errors["gold_head_pos"].append(tok_gold["upos"])
                    errors["pred_head_pos"].append(tok_pred["upos"])
                    errors["gold_head"].append(tok_gold["head"])
                    errors["pred_head"].append(tok_pred["head"])
                    errors["gold_deprel"].append(tok_gold["deprel"])
                    errors["pred_deprel"].append(tok_pred["deprel"])

                # if wrong_head and not wrong_deprel:
                #     errors["gold_head"].append(tok_gold["head"])
                #     errors["pred_head"].append(tok_pred["head"])
                #     errors["gold_deprel"].append("na")
                #     errors["pred_deprel"].append("na")
                # elif not wrong_head and wrong_deprel:
                #     errors["gold_head"].append("na")
                #     errors["pred_head"].append("na")
                #     errors["gold_deprel"].append(tok_gold["deprel"])
                #     errors["pred_deprel"].append(tok_pred["deprel"])
                # elif wrong_head and wrong_deprel:
                #     errors["gold_head"].append(tok_gold["head"])
                #     errors["pred_head"].append(tok_pred["head"])
                #     errors["gold_deprel"].append(tok_gold["deprel"])
                #     errors["pred_deprel"].append(tok_pred["deprel"])
    return errors



In [50]:
gold_const_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
pred_const_file = os.path.join(PROJECT_ROOT, "predictions", "stanza", "lang=en,bert=finetune,charlm=yes,pretrain=yes,epochs=100,deprojz=yes,matched=yes.conllu")
const_errors = find_errors(gold_const_file, pred_const_file)

In [51]:
const_err_df = pd.DataFrame(const_errors)
const_err_df

,sent_id,token_id,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-2,23,21,ADJ,19,ADJ,dep,dep
1,english-penn-test-4,11,5,PUNCT,13,PUNCT,punct,punct
2,english-penn-test-4,26,5,PUNCT,13,PUNCT,punct,punct
3,english-penn-test-5,7,6,ADP,6,ADP,advmod,compound:prt
4,english-penn-test-5,10,7,NOUN,6,NOUN,nmod,nmod
...,...,...,...,...,...,...,...,...
2441,english-penn-test-2414,10,11,NUM,11,NUM,compound,nummod
2442,english-penn-test-2414,16,3,PUNCT,8,PUNCT,punct,punct
2443,english-penn-test-2414,17,20,VERB,8,VERB,case,xcomp
2444,english-penn-test-2414,20,3,NOUN,17,NOUN,nmod,dobj


In [53]:
abs(const_err_df["gold_head"] - const_err_df["token_id"]).mean()

np.float64(5.155764513491414)

In [54]:
abs(const_err_df["pred_head"] - const_err_df["gold_head"]).mean()

np.float64(4.439493049877351)

In [56]:
gold_dep_file = os.path.join(PROJECT_ROOT, "data", "raw", "UD_English-Penn", "en_penn-ud-test.conllu")
pred_dep_file = os.path.join(PROJECT_ROOT, "predictions", "depparse", "lang=en,bert=frozen,charlm=no,pretrain=no.conllu")
dep_errors = find_errors(gold_dep_file, pred_dep_file)

In [57]:
dep_err_df = pd.DataFrame(dep_errors)
dep_err_df

,sent_id,token_id,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-2,22,23,PUNCT,19,PUNCT,punct,punct
1,english-penn-test-2,23,21,ADJ,19,ADJ,dep,nsubj
2,english-penn-test-2,29,23,NOUN,19,NOUN,nmod,nmod
3,english-penn-test-2,30,23,PUNCT,33,PUNCT,punct,punct
4,english-penn-test-2,37,35,NOUN,35,NOUN,dobj,nmod:tmod
...,...,...,...,...,...,...,...,...
4509,english-penn-test-2413,14,4,NOUN,6,NOUN,nmod,nmod
4510,english-penn-test-2414,16,3,PUNCT,8,PUNCT,punct,punct
4511,english-penn-test-2414,17,20,VERB,8,VERB,case,xcomp
4512,english-penn-test-2414,20,3,NOUN,17,NOUN,nmod,dobj


In [58]:
abs(dep_err_df["gold_head"] - dep_err_df["token_id"]).mean()

np.float64(5.798848028356225)

In [65]:
dep_wrong_head_df = dep_err_df.query("gold_head != pred_head")
dep_wrong_head_df 

,sent_id,token_id,gold_head,gold_head_pos,pred_head,pred_head_pos,gold_deprel,pred_deprel
0,english-penn-test-2,22,23,PUNCT,19,PUNCT,punct,punct
1,english-penn-test-2,23,21,ADJ,19,ADJ,dep,nsubj
2,english-penn-test-2,29,23,NOUN,19,NOUN,nmod,nmod
3,english-penn-test-2,30,23,PUNCT,33,PUNCT,punct,punct
5,english-penn-test-3,12,0,VERB,18,VERB,root,advcl
...,...,...,...,...,...,...,...,...
4508,english-penn-test-2409,32,8,PUNCT,26,PUNCT,punct,punct
4509,english-penn-test-2413,14,4,NOUN,6,NOUN,nmod,nmod
4510,english-penn-test-2414,16,3,PUNCT,8,PUNCT,punct,punct
4511,english-penn-test-2414,17,20,VERB,8,VERB,case,xcomp


In [67]:
abs(dep_wrong_head_df["gold_head"] - dep_wrong_head_df["token_id"]).mean()

np.float64(6.0470752089136495)

In [68]:
abs(dep_wrong_head_df["pred_head"] - dep_wrong_head_df["token_id"]).mean()

np.float64(5.350974930362117)

In [60]:
abs(dep_err_df["pred_head"] - dep_err_df["gold_head"]).mean()

np.float64(4.523925564909171)

In [43]:
const_pos_count = const_err_df.groupby(["gold_head_pos", "pred_head_pos"])["token_id"].count().sort_values(ascending=False)
const_pos_count

gold_head_pos  pred_head_pos
NOUN           NOUN             518
PUNCT          PUNCT            492
VERB           VERB             331
PROPN          PROPN            241
ADV            ADV              201
ADJ            ADJ              147
ADP            ADP              134
NUM            NUM              108
DET            DET               74
CONJ           CONJ              61
SYM            SYM               57
PRON           PRON              38
AUX            AUX               12
PART           PART              12
SCONJ          SCONJ              9
X              X                  9
INTJ           INTJ               2
Name: token_id, dtype: int64

In [44]:
dep_pos_count = dep_err_df.groupby(["gold_head_pos", "pred_head_pos"])["token_id"].count().sort_values(ascending=False)
dep_pos_count

gold_head_pos  pred_head_pos
PUNCT          PUNCT            937
NOUN           NOUN             927
VERB           VERB             773
PROPN          PROPN            380
ADV            ADV              313
ADJ            ADJ              258
ADP            ADP              230
NUM            NUM              170
CONJ           CONJ             147
DET            DET              107
SYM            SYM               87
PRON           PRON              83
PART           PART              32
SCONJ          SCONJ             25
AUX            AUX               25
X              X                 12
INTJ           INTJ               8
Name: token_id, dtype: int64